In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path
# 사용: p = Path("data/file.csv")
import math
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
# 1) 파일 경로 지정 (여기만 바꾸세요)

from IPython.display import display, HTML
# 1) 한글 폰트 지정
# macOS
plt.rcParams["font.family"] = "AppleGothic"
# Windows
# plt.rcParams["font.family"] = "Malgun Gothic"
# Linux(우분투 등, 나눔)
# plt.rcParams["font.family"] = "NanumGothic"

# 2) 마이너스 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
# ================== 1) 파라미터 ==================
CRS_M   = 5179      # 작업/거리계산용(미터)
CRS_LAT = 4326      # 지도 표시용
R       = 200       # 육각형 반경(m) (center→vertex)
MARGIN  = 0         # '온전한' 판정 여유(경계 내부로 MARGIN m 수축, 0이면 정확히 within)

# 선택옵션 -----------------------------
USE_FULL_WITHIN = False   # True: "온전한(완전 포함)" 육각형만 사용 / False: 센트로이드 기준 사용
CLIP_TO_BOUND   = True    # True: 경계로 육각형을 클립하여 라인 생성 / False: 원형 그대로 라인
CENTER_MARGIN   = 0       # 센트로이드 기준일 때, 경계에서 최소 이격(m). 0이면 경계 안이면 허용
# ---------------------------------------


shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')
gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 경계 데이터
dunsan = gdf[gdf['EMD_NM'] == '둔산동'].copy()
if dunsan.empty:
    raise ValueError("경계에서 '둔산동'이 발견되지 않았습니다. EMD_NM 값을 확인하세요.")

gdf_5179 = dunsan.to_crs(CRS_M)
gdf_4326 = dunsan.to_crs(CRS_LAT)

In [ ]:
def folium_deepnote_show(m):
    data = m.get_root().render()
    data_fixed_height = data.replace('width: 100%;height: 100%', 'width: 100%').replace('height: 100.0%;', 'height: 609px;', 1)
    display(HTML(data_fixed_height))

In [ ]:
import geopandas as gpd
import pandas as pd


stores_df = pd.read_csv("./data/1차필터링.csv")
df = stores_df.copy()  # <- 네 데이터프레임 이름

# 1) 숫자형 보정(문자/결측 방지)
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df = df.dropna(subset=["lon","lat"])

# 2) GeoDataFrame으로: EPSG:4326(경위도) -> 5179(미터)
gdf_4326 = gpd.GeoDataFrame(
    df.reset_index(drop=True),
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=4326
)
supply_gdf = gdf_4326.to_crs(5179).copy()
supply_gdf


supply_gdf["s_id"] = supply_gdf.index.astype(int)

demand_gdf = pd.read_csv("./data/수요지데이터.csv")
demand_gdf

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt

def make_demand_gdf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    df = df.copy()

    # 1) 가중치 컬럼 정리: suffix_aw_sum -> weight
    if "suffix_aw_sum" in df.columns:
        df["weight"] = pd.to_numeric(df["suffix_aw_sum"], errors="coerce").fillna(0)
    elif "pop" in df.columns:
        df["weight"] = pd.to_numeric(df["pop"], errors="coerce").fillna(0)
    else:
        raise ValueError("가중치로 쓸 컬럼(suffix_aw_sum 또는 pop)이 필요합니다.")

    # 2) geometry 만들기 (우선순위: geometry(WKT/Point) → lon/lat)
    gdf_4326 = None

    if "geometry" in df.columns:
        # (a) 이미 shapely Point 인가?
        try:
            is_geom = df["geometry"].apply(lambda v: hasattr(v, "geom_type")).all()
        except Exception:
            is_geom = False

        if is_geom:
            gdf_4326 = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)
        else:
            # (b) WKT 문자열인가? ("POINT (.. ..)")
            as_str = df["geometry"].astype(str)
            if as_str.str.startswith("POINT").all():
                df["geometry"] = as_str.apply(wkt.loads)
                gdf_4326 = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)

    if gdf_4326 is None:
        # (c) lon/lat로부터 생성
        if not {"lon","lat"}.issubset(df.columns):
            raise ValueError("geometry가 없으면 lon/lat 컬럼이 필요합니다.")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df = df.dropna(subset=["lon","lat"]).copy()
        gdf_4326 = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["lon"], df["lat"]),
            crs=4326
        )

    # 3) 5179(미터)로 변환 + ID 부여 + 클린업
    gdf = gdf_4326.to_crs(5179).reset_index(drop=True)
    gdf["d_id"] = gdf.index.astype(int)
    gdf["weight"] = gdf["weight"].clip(lower=0)  # 음수 방지

    # (선택) 가중치 0인 셀 제거
    # gdf = gdf[gdf["weight"] > 0].copy()

    # 최종 필요한 컬럼만 남겨도 됨
    keep_cols = ["d_id", "geometry", "weight"]
    if "hex_id" in gdf.columns: keep_cols.insert(1, "hex_id")
    return gdf[keep_cols]

# 사용 예시
# df = pd.read_csv("...")  # 지금 가진 DataFrame
# demand_gdf = make_demand_gdf(df)
# demand_gdf.head()
demand_gdf = make_demand_gdf(demand_gdf)
demand_gdf

In [ ]:
supply_df = supply_gdf[['s_id', 'store_name', 'cluster', 'geometry']]
supply_df.head()
demand_gdf.head()

In [ ]:
demand_gdf

In [ ]:
demand_gdf.sort_values("weight", ascending=False).head(30)
demand_gdf['ratio'] = demand_gdf['weight'] / demand_gdf['weight'].sum()
demand_gdf

In [ ]:
import pandas as pd

def build_cover_edges(demand_gdf, supply_gdf, R_default=350.0):
    # 후보별 R 컬럼이 없으면 상수 반경 사용
    if "R" not in supply_gdf.columns:
        supply_gdf = supply_gdf.assign(R=R_default)

    sidx = supply_gdf.sindex
    rows = []
    Rmax = float(supply_gdf["R"].max())

    for d_id, dgeom in demand_gdf[["d_id","geometry"]].itertuples(index=False):
        cand_idx = list(sidx.intersection(dgeom.buffer(Rmax).bounds))
        if not cand_idx:
            continue
        cand = supply_gdf.iloc[cand_idx].copy()
        dist = cand.geometry.distance(dgeom)
        ok = cand[dist <= cand["R"]]
        if not ok.empty:
            for s_id, dist_m in zip(ok["s_id"], dist[dist <= cand["R"]]):
                rows.append((int(d_id), int(s_id), float(dist_m)))

    return pd.DataFrame(rows, columns=["d_id","s_id","dist_m"])


In [ ]:
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD
import numpy as np

# pip install pulp  (한 번만)
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD
import numpy as np

def solve_mclp(demand_gdf, supply_gdf, cover_df, p):
    w = demand_gdf.set_index("d_id")["weight"].to_dict()
    demand_ids = list(w.keys())
    supply_ids = supply_gdf["s_id"].astype(int).tolist()

    # N(i): i를 커버 가능한 공급지 집합
    Ni = {i: set() for i in demand_ids}
    for d_id, s_id, _ in cover_df.itertuples(index=False):
        Ni[d_id].add(int(s_id))

    model = LpProblem("MCLP", LpMaximize)
    x = {j: LpVariable(f"x_{j}", 0, 1, LpBinary) for j in supply_ids}
    z = {i: LpVariable(f"z_{i}", 0, 1, LpBinary) for i in demand_ids}

    model += lpSum(w[i]*z[i] for i in demand_ids)                 # 목적
    model += lpSum(x[j] for j in supply_ids) == p                 # 시설 수
    for i in demand_ids:                                          # 커버 제약
        if len(Ni[i]) == 0:
            model += z[i] == 0
        else:
            model += z[i] <= lpSum(x[j] for j in Ni[i])

    model.solve(PULP_CBC_CMD(msg=False))

    chosen  = [j for j in supply_ids if x[j].value() > 0.5]
    covered = [i for i in demand_ids if z[i].value() > 0.5]
    obj   = float(sum(w[i] for i in covered))
    total = float(sum(w.values()))
    return {
        "chosen_s_ids": chosen,
        "covered_ratio": obj/total if total>0 else np.nan,
        "covered_sum": obj,
        "total_weight": total,
        "covered_d_ids": covered
    }

In [ ]:
# R: 150m ~ 450m 까지 25~50m 간격 예시 (원하는 대로 조정 가능)
R_LIST = [200, 250, 300, 350, 400]

# p: 5 ~ 15개 (원하는 범위로 늘리거나 줄여도 OK)
P_LIST = list(range(6, 19))


def run_scenarios(demand_gdf, supply_gdf, R_list, p_list):
    results, details = [], {}
    for R in R_list:
        cover_df = build_cover_edges(demand_gdf, supply_gdf, R_default=R)
        for p in p_list:
            if cover_df.empty:
                res = {"chosen_s_ids":[], "covered_ratio":0.0,
                       "covered_sum":0.0, "total_weight": float(demand_gdf["weight"].sum()),
                       "covered_d_ids":[]}
            else:
                res = solve_mclp(demand_gdf, supply_gdf, cover_df, p=p)
            results.append({
                "R": R, "p": p,
                "covered_pct": round(res["covered_ratio"]*100, 2) if res["total_weight"]>0 else 0.0,
                "chosen_count": len(res["chosen_s_ids"])
            })
            details[(R,p)] = res
    return pd.DataFrame(results).sort_values(["R","p"]).reset_index(drop=True), details

summary_df, detail_map = run_scenarios(demand_gdf, supply_gdf, R_LIST, P_LIST)
summary_df.to_csv('./data/최적화_시나리오_요약.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

def _covered_stats(demand_gdf, covered_d_ids):
    """개수/가중치 기반 커버 통계를 한 번에 계산."""
    covered_set = set(map(int, covered_d_ids))
    total_n = len(demand_gdf)
    total_w = float(demand_gdf["weight"].sum())

    # 개수 기반
    covered_n = int(demand_gdf["d_id"].astype(int).isin(covered_set).sum())
    ratio_n = covered_n / total_n if total_n > 0 else np.nan

    # 가중치 기반
    w_map = demand_gdf.set_index("d_id")["weight"]
    covered_w = float(w_map[w_map.index.astype(int).isin(covered_set)].sum())
    ratio_w = covered_w / total_w if total_w > 0 else np.nan

    return {
        "covered_count": covered_n,
        "total_count": total_n,
        "covered_ratio_n": ratio_n,        # 개수 기반 커버율
        "covered_sum": covered_w,
        "total_weight": total_w,
        "covered_ratio_w": ratio_w,        # 가중치 기반 커버율
        "uncovered_count": total_n - covered_n,
        "uncovered_weight_sum": total_w - covered_w,
        "uncovered_ratio_n": 1 - ratio_n if pd.notna(ratio_n) else np.nan,
        "uncovered_ratio_w": 1 - ratio_w if pd.notna(ratio_w) else np.nan,
    }

def run_scenarios(demand_gdf, supply_gdf, R_list, p_list):
    results, details = [], {}

    # 안전장치: 필요한 컬럼 확인
    for c in ["d_id","weight","geometry"]:
        assert c in demand_gdf.columns, f"demand_gdf에 '{c}'가 필요합니다."
    for c in ["s_id","geometry"]:
        assert c in supply_gdf.columns, f"supply_gdf에 '{c}'가 필요합니다."

    for R in R_list:
        cover_df = build_cover_edges(demand_gdf, supply_gdf, R_default=R)

        for p in p_list:
            if cover_df.empty:
                # 아무것도 커버하지 못하는 경우
                covered_d_ids = []
                chosen_s_ids = []
            else:
                res = solve_mclp(demand_gdf, supply_gdf, cover_df, p=p)
                covered_d_ids = res["covered_d_ids"]
                chosen_s_ids  = res["chosen_s_ids"]

            st = _covered_stats(demand_gdf, covered_d_ids)

            # 요약 행(개수/가중치 둘 다 기록)
            results.append({
                "R": R, "p": p,
                # 개수 기반
                "covered_ratio_n": st["covered_ratio_n"],
                "covered_pct_n": round(st["covered_ratio_n"]*100, 2) if pd.notna(st["covered_ratio_n"]) else np.nan,
                "covered_count": st["covered_count"],
                "total_count": st["total_count"],
                "uncovered_ratio_n": st["uncovered_ratio_n"],
                "uncovered_count": st["uncovered_count"],

                # 가중치(인구) 기반
                "covered_ratio_w": st["covered_ratio_w"],
                "covered_pct_w": round(st["covered_ratio_w"]*100, 2) if pd.notna(st["covered_ratio_w"]) else np.nan,
                "covered_sum": st["covered_sum"],
                "total_weight": st["total_weight"],
                "uncovered_ratio_w": st["uncovered_ratio_w"],
                "uncovered_weight_sum": st["uncovered_weight_sum"],

                # 참고
                "chosen_count": len(chosen_s_ids),
            })

            details[(R,p)] = {
                "chosen_s_ids": chosen_s_ids,
                "covered_d_ids": covered_d_ids,
                **st,  # 위 통계 전부 포함
            }

    summary_df = pd.DataFrame(results).sort_values(["R","p"]).reset_index(drop=True)
    return summary_df, details
summary_df, detail_map = run_scenarios(
    demand_gdf=demand_gdf,
    supply_gdf=supply_gdf,
    R_list=R_LIST,
    p_list=P_LIST
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

R_LIST = [200, 250, 300, 350, 400]  # 존재하는 값만 자동으로 그려짐

df = summary_df.copy()

plt.figure(figsize = (14, 8))
for R in R_LIST:
    gi = df[np.isclose(df["R"], R)].sort_values("p")
    if gi.empty: 
        continue
    plt.plot(gi["p"], gi["covered_pct_w"], marker="o", label=f"R={int(R)}m")
plt.xlabel("p (시설 수)", fontsize=20)
plt.ylabel("커버율 (%)", fontsize=20)
plt.title("R별 커버율 vs p", fontsize=24)
plt.legend(fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

R_LIST = [200, 250, 300, 350, 400]  # 존재하는 값만 자동으로 그려짐

df = summary_df.copy()

plt.figure(figsize = (14, 8))
for R in R_LIST:
    gi = df[np.isclose(df["R"], R)].sort_values("p")
    if gi.empty: 
        continue
    plt.plot(gi["p"], gi["covered_pct_n"], marker="o", label=f"R={int(R)}m")
plt.xlabel("p (시설 수)")
plt.ylabel("커버율 (%)")
plt.title("R별 커버율 vs p")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
R_chosen, p_chosen = 350.0, 11
res = detail_map[(R_chosen, p_chosen)]

# 선택된 공급지/커버 여부 플래그
supply_sel = supply_gdf.assign(selected=supply_gdf["s_id"].isin(res["chosen_s_ids"]).astype(int))
demand_cov = demand_gdf.assign(covered=demand_gdf["d_id"].isin(res["covered_d_ids"]).astype(int))

print("선정 공급지:", res["chosen_s_ids"])
print("커버율:", round(res["covered_ratio_w"]*100, 2), "%")

# 미커버 상위 수요 확인
uncovered_top = demand_cov.query("covered == 0").sort_values("weight", ascending=False).head(20)
uncovered_top[["d_id","weight","ratio"]]


In [ ]:
import pandas as pd
import numpy as np

R_TARGET = 350.0
P_TARGET = 10

# 0) 시나리오 결과가 없으면 즉석에서 풀이
try:
    res = detail_map[(R_TARGET, P_TARGET)]
except Exception:
    cover_df = build_cover_edges(demand_gdf, supply_gdf, R_default=R_TARGET)
    res = solve_mclp(demand_gdf, supply_gdf, cover_df, p=P_TARGET)

chosen_s = set(map(int, res["chosen_s_ids"]))
covered_d = set(map(int, res["covered_d_ids"]))

# 1) 이번 R에서의 커버 엣지 생성
cover_df = build_cover_edges(demand_gdf, supply_gdf, R_default=R_TARGET)

# 2) “선정된 공급지 ↔ 커버된 수요지” 쌍만 필터
cov_pairs = (
    cover_df[cover_df["s_id"].isin(chosen_s) & cover_df["d_id"].isin(covered_d)]
    .merge(demand_gdf[["d_id","weight"]], on="d_id", how="left")
    .sort_values(["s_id","weight"], ascending=[True, False])
    .reset_index(drop=True)
)
# columns: d_id, s_id, dist_m(유클리드), weight

# 3) 수요지별 ‘가장 가까운’ 선정 공급지에 1:1 배정(선택)
idx = cov_pairs.groupby("d_id")["dist_m"].idxmin()
assigned_pairs = cov_pairs.loc[idx, ["d_id","s_id","dist_m","weight"]].reset_index(drop=True)

# 4) 공급지별 요약(커버 기준 / 배정 기준)
sum_cover = (cov_pairs.groupby("s_id")
             .agg(covered_demands=("d_id","nunique"),
                  covered_weight=("weight","sum"),
                  avg_dist_m=("dist_m","mean"))
             .reset_index()
             .sort_values(["covered_weight","covered_demands"], ascending=[False, False]))

sum_assign = (assigned_pairs.groupby("s_id")
              .agg(assigned_demands=("d_id","nunique"),
                   assigned_weight=("weight","sum"),
                   avg_assigned_dist_m=("dist_m","mean"))
              .reset_index()
              .sort_values(["assigned_weight","assigned_demands"], ascending=[False, False]))


In [ ]:
# === 0) 준비: 미터 좌표계로 변환 ===
CRS_M = 5179
sup_m = supply_gdf.to_crs(CRS_M).copy()
dem_m = demand_gdf.to_crs(CRS_M).copy()

chosen    = set(res["chosen_s_ids"])
covered_d = set(res["covered_d_ids"])

sup_sel_m = sup_m[sup_m["s_id"].isin(chosen)].copy()
dem_cov_m = dem_m[dem_m["d_id"].isin(covered_d)].copy()  # 커버된 수요지만 대상

# === 1) 반경 R 내 many-to-many 커버 페어 생성 ===
sup_buf = sup_sel_m[["s_id", "geometry"]].copy()
sup_buf["geometry"] = sup_buf.geometry.buffer(R_chosen)

pairs = gpd.sjoin(
    dem_cov_m[["d_id","geometry","weight"]],   # 수요(점)
    sup_buf,                                   # 공급(버퍼)
    how="inner",
    predicate="within"
).drop(columns=["index_right"])                # right의 인덱스 컬럼 정리

# === 2) 실제 거리 계산(수요점 ↔ 공급 '점' 사이) ===
pairs = pairs.merge(
    sup_sel_m[["s_id","geometry"]].rename(columns={"geometry":"s_geom"}),
    on="s_id", how="left"
)
pairs["dist_m"] = pairs.geometry.distance(pairs["s_geom"])

# 최종 edge 테이블(모든 커버 관계)
edge_df_all = pairs.rename(columns={"geometry":"d_geom"})[["s_id","d_id","dist_m","weight"]]

# === 3) (선택) 수요별 '가장 가까운' 선정 공급지 1:1 매핑 버전 ===
edge_df_nearest = (edge_df_all
    .sort_values(["d_id","dist_m"])
    .drop_duplicates(subset="d_id")
    .reset_index(drop=True))

# === 4) 공급지별 요약표(몇 개, 얼마만큼, 얼마나 멀리) ===
per_supply = (edge_df_all.groupby("s_id")
    .agg(n_demand=("d_id","nunique"),
         covered_pop_sum=("weight","sum"),
         avg_dist_m=("dist_m","mean"),
         median_dist_m=("dist_m","median"),
         max_dist_m=("dist_m","max"))
    .reset_index())


In [ ]:
supply_df

In [ ]:
df = per_supply[['s_id', 'n_demand', 'covered_pop_sum', 'avg_dist_m']]
df2 = pd.merge(df, supply_df, on='s_id')
df3 = df2[['store_name', 'n_demand', 'covered_pop_sum', 'avg_dist_m']]
df3

In [ ]:
import folium

# 1) 파일 경로 지정 (여기만 바꾸세요)
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)
dunsan_4326

dunsan_5179 = dunsan.to_crs(5179)   


b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m

In [ ]:
import folium
import geopandas as gpd

# 좌표 변환
supply_wgs = supply_gdf.to_crs(4326).copy()
demand_wgs = demand_gdf.to_crs(4326).copy()
supply_wgs["selected"] = supply_wgs["s_id"].astype(int).isin(chosen_s).astype(int)
demand_wgs["covered"]  = demand_wgs["d_id"].astype(int).isin(covered_d).astype(int)

# 지도 베이스

# 선정 공급지 서비스권(원)
for row in supply_wgs.itertuples():
    if row.selected == 1:
        folium.Circle(
            location=(row.geometry.y, row.geometry.x),
            radius=float(R_TARGET), color="#2E7D32", fill=True, fill_opacity=0.08, weight=1
        ).add_to(m)

# 공급지 마커(선정=진한)
for row in supply_wgs.itertuples():
    color = "#2E7D32" if row.selected==1 else "#9E9E9E"
    folium.CircleMarker((row.geometry.y, row.geometry.x), radius=5,
                        color=color, fill=True, fill_opacity=1,
                        tooltip=f"s_id={row.s_id}").add_to(m)

# 수요지 마커(커버=파랑, 미커버=빨강)
for row in demand_wgs.itertuples():
    color = "#1976D2" if row.covered==1 else "#D32F2F"
    folium.CircleMarker((row.geometry.y, row.geometry.x), radius=3,
                        color=color, fill=True, fill_opacity=0.9,
                        tooltip=f"d_id={row.d_id}, weight={float(row.weight):,.1f}").add_to(m)

# 최근접 배정된 쌍만 선으로 연결 (너무 많을 수 있으니 상위 가중치 위주로 제한)
s_coords = supply_wgs[["s_id","geometry"]].rename(columns={"geometry":"s_geom"})
d_coords = demand_wgs[["d_id","geometry"]].rename(columns={"geometry":"d_geom"})
ap = assigned_pairs.merge(s_coords, on="s_id").merge(d_coords, on="d_id")\
                   .sort_values("weight", ascending=False)

max_lines = 400                # 전체 선 개수 제한
topk_per_supply = 30           # 공급지별 최대 선 개수
drawn, per_supply = 0, {}

for r in ap.itertuples():
    sid = int(r.s_id)
    per_supply.setdefault(sid, 0)
    if per_supply[sid] >= topk_per_supply:
        continue
    folium.PolyLine([(r.s_geom.y, r.s_geom.x), (r.d_geom.y, r.d_geom.x)],
                    weight=1, opacity=0.7).add_to(m)
    per_supply[sid] += 1
    drawn += 1
    if drawn >= max_lines:
        break

m

In [ ]:
supply_gdf.loc[res['chosen_s_ids']][['store_name', 'cluster', 'lon', 'lat']]